# 05B — S3E + Cosine-Utility MBR — Fragment-Shot Prompting (Exact Locked) — FINAL V1

Notebook ini memproses **265 kandidat final (53 × 5)** dari skenario **Fragment-Shot Prompting (Exact Locked)**.

Notebook ini:
- tidak menjalankan Llama lagi;
- tidak membaca reference Bahasa Indonesia;
- hanya memvalidasi kandidat, membentuk embedding MPNet, menghitung cosine similarity, S3E, dan MBR;
- belum menghitung BLEU/chrF++.

**S3E**  
`H = -(1/n) Σ_i log[(1/(n-1)) Σ_(j≠i) S(y_i,y_j)]`

**MBR utility**  
`U(y_i) = (1/(n-1)) Σ_(j≠i) S(y_i,y_j)`

Ketentuan:
- model embedding: `sentence-transformers/paraphrase-multilingual-mpnet-base-v2`;
- embedding L2-normalized, dimensi 768;
- cosine negatif → 0;
- diagonal disimpan 1 tetapi tidak dipakai dalam mean;
- `epsilon=1e-12`;
- tie MBR → candidate index terkecil;
- S3E berjalan paralel dengan MBR dan tidak menjadi input seleksi.

Output:
`Hasil/final_selection/fsp_locked/<run_input>/`

## 1. Dependency audit — tanpa pip install

In [1]:
from importlib.metadata import version, PackageNotFoundError
from packaging.version import Version

EXPECTED = {
    "sentence-transformers": "3.4.1",
    "transformers": "4.48.1",
}

errors = []

for package_name, expected in EXPECTED.items():
    try:
        actual = version(package_name)
    except PackageNotFoundError:
        errors.append(
            f"{package_name}: NOT INSTALLED. "
            f"Install sekali di terminal: "
            f"/venv/main/bin/python -m pip install "
            f"'{package_name}=={expected}'"
        )
        continue

    ok = Version(actual) == Version(expected)
    print(
        f"{package_name:22} {actual:12} "
        f"{'OK' if ok else 'MISMATCH'}"
    )

    if not ok:
        errors.append(
            f"{package_name}: actual={actual}, expected={expected}"
        )

if errors:
    raise RuntimeError(
        "Dependency audit gagal:\n- "
        + "\n- ".join(errors)
    )

print("\nDEPENDENCY AUDIT: PASS")

sentence-transformers  3.4.1        OK
transformers           4.48.1       OK

DEPENDENCY AUDIT: PASS


## 2. Environment

In [2]:
from __future__ import annotations

import gc
import hashlib
import json
import os
import platform
import re
from datetime import datetime, timezone
from pathlib import Path

import numpy as np
import pandas as pd
import torch
from IPython.display import display
from sentence_transformers import SentenceTransformer

PROJECT_ROOT = Path(
    os.getenv(
        "SKRIPSI_PROJECT_ROOT",
        "/workspace/10122233_CikitaMeydilvira_Skripsi",
    )
).expanduser().resolve()

HF_HOME = Path(
    os.getenv(
        "HF_HOME",
        "/workspace/.hf_home",
    )
).expanduser().resolve()

EMBEDDING_CACHE = (
    HF_HOME / "sentence_transformers"
).resolve()

EMBEDDING_MODEL_ID = (
    "sentence-transformers/"
    "paraphrase-multilingual-mpnet-base-v2"
)

SCENARIO_SLUG = "fsp_locked"
SCENARIO_LABEL = "Fragment-Shot Prompting (Exact Locked)"
INPUT_SCENARIO_DIR = (
    PROJECT_ROOT
    / "Hasil"
    / "final_generation"
    / "fsp_locked"
)
OUTPUT_ROOT = (
    PROJECT_ROOT
    / "Hasil"
    / "final_selection"
    / SCENARIO_SLUG
)

EXPECTED_TEST_ROWS = 53
EXPECTED_TOTAL_CANDIDATES = 265
EXPECTED_SEEDS = {42, 43, 44, 45, 46}
EPS = 1e-12
TIE_TOL = 1e-12
EMBEDDING_BATCH_SIZE = 32

os.environ["HF_HOME"] = str(HF_HOME)
os.environ.setdefault(
    "TOKENIZERS_PARALLELISM",
    "false",
)

EMBEDDING_CACHE.mkdir(
    parents=True,
    exist_ok=True,
)

if not PROJECT_ROOT.exists():
    raise FileNotFoundError(
        f"PROJECT_ROOT tidak ditemukan: {PROJECT_ROOT}"
    )

if not INPUT_SCENARIO_DIR.exists():
    raise FileNotFoundError(
        f"Folder final generation tidak ditemukan: "
        f"{INPUT_SCENARIO_DIR}"
    )

if torch.cuda.is_available():
    EMBEDDING_DEVICE = "cuda"
    free_b, total_b = torch.cuda.mem_get_info(0)
    print(
        "GPU:",
        torch.cuda.get_device_name(0),
        f"| free={free_b/(1024**3):.2f} GiB",
    )
else:
    EMBEDDING_DEVICE = "cpu"
    print("CUDA tidak tersedia; MPNet memakai CPU.")

print("PROJECT_ROOT      :", PROJECT_ROOT)
print("INPUT_SCENARIO_DIR:", INPUT_SCENARIO_DIR)
print("OUTPUT_ROOT       :", OUTPUT_ROOT)
print("EMBEDDING_DEVICE  :", EMBEDDING_DEVICE)
print("EMBEDDING_CACHE   :", EMBEDDING_CACHE)

GPU: NVIDIA GeForce RTX 4090 | free=22.66 GiB
PROJECT_ROOT      : /workspace/10122233_CikitaMeydilvira_Skripsi
INPUT_SCENARIO_DIR: /workspace/10122233_CikitaMeydilvira_Skripsi/Hasil/final_generation/fsp_locked
OUTPUT_ROOT       : /workspace/10122233_CikitaMeydilvira_Skripsi/Hasil/final_selection/fsp_locked
EMBEDDING_DEVICE  : cuda
EMBEDDING_CACHE   : /workspace/.hf_home/sentence_transformers


## 3. Temukan + audit kandidat final

In [3]:
FORBIDDEN_REFERENCE_COLUMNS = {
    "reference",
    "reference_text",
    "indonesia_text",
    "gold",
    "gold_reference",
}

def read_json_if_exists(path: Path):
    if not path.exists():
        return None
    try:
        return json.loads(
            path.read_text(encoding="utf-8")
        )
    except Exception:
        return None


def canonicalize(path: Path) -> pd.DataFrame:
    raw = pd.read_csv(
        path,
        dtype={"row_id": str},
        keep_default_na=False,
        encoding="utf-8-sig",
    )

    leaked = (
        FORBIDDEN_REFERENCE_COLUMNS
        & set(raw.columns)
    )
    if leaked:
        raise RuntimeError(
            f"Reference column ditemukan: {sorted(leaked)}"
        )

    if "classic" == "classic":
        required = {
            "scenario",
            "row_id",
            "candidate_id",
            "seed",
            "source",
            "output",
        }
        missing = required - set(raw.columns)
        if missing:
            raise RuntimeError(
                f"Kolom kurang: {sorted(missing)}"
            )

        df = pd.DataFrame({
            "scenario": raw["scenario"].astype(str),
            "row_id": raw["row_id"].astype(str),
            "conversation_no":
                raw["conversation_no"].astype(str)
                if "conversation_no" in raw.columns else "",
            "turn_order":
                raw["turn_order"].astype(str)
                if "turn_order" in raw.columns else "",
            "source_text": raw["source"].astype(str),
            "candidate_index":
                pd.to_numeric(
                    raw["candidate_id"],
                    errors="raise",
                ).astype(int),
            "seed":
                pd.to_numeric(
                    raw["seed"],
                    errors="raise",
                ).astype(int),
            "candidate_text": raw["output"].astype(str),
            "technical_error":
                raw["error"].astype(str)
                if "error" in raw.columns else "",
        })

        for col in [
            "temperature",
            "top_p",
            "top_k",
            "max_new_tokens",
            "coverage",
            "fragment_count",
            "example_count",
            "max_fragment_len",
            "k_per_fragment",
            "prompt_tokens",
            "completion_tokens",
            "latency_seconds",
        ]:
            if col in raw.columns:
                df[col] = raw[col].values

    else:
        required = {
            "scenario",
            "row_id",
            "candidate_index",
            "seed",
            "duri_text",
            "candidate_text",
        }
        missing = required - set(raw.columns)
        if missing:
            raise RuntimeError(
                f"Kolom kurang: {sorted(missing)}"
            )

        df = pd.DataFrame({
            "scenario": raw["scenario"].astype(str),
            "row_id": raw["row_id"].astype(str),
            "conversation_no":
                raw["conversation_no"].astype(str)
                if "conversation_no" in raw.columns else "",
            "turn_order":
                raw["turn_order"].astype(str)
                if "turn_order" in raw.columns else "",
            "source_text": raw["duri_text"].astype(str),
            "candidate_index":
                pd.to_numeric(
                    raw["candidate_index"],
                    errors="raise",
                ).astype(int),
            "seed":
                pd.to_numeric(
                    raw["seed"],
                    errors="raise",
                ).astype(int),
            "candidate_text":
                raw["candidate_text"].astype(str),
            "technical_error": "",
        })

        for col in [
            "temperature",
            "top_p",
            "top_k",
            "max_new_tokens",
            "generation_time_seconds",
        ]:
            if col in raw.columns:
                df[col] = raw[col].values

        if "is_empty" in raw.columns:
            empty_mask = (
                raw["is_empty"]
                .astype(str)
                .str.casefold()
                .isin({"true", "1", "yes"})
            )
            df.loc[
                empty_mask,
                "technical_error",
            ] = "empty_output"

    return df


def audit_candidates(df: pd.DataFrame):
    if len(df) != EXPECTED_TOTAL_CANDIDATES:
        raise RuntimeError(
            f"Jumlah kandidat {len(df)} != 265."
        )

    if df["row_id"].nunique() != EXPECTED_TEST_ROWS:
        raise RuntimeError(
            f"Unique row_id {df['row_id'].nunique()} != 53."
        )

    counts = df.groupby("row_id").size()
    if not (counts == 5).all():
        raise RuntimeError(
            "Tidak semua row_id memiliki 5 kandidat."
        )

    if set(df["seed"].astype(int).unique()) != EXPECTED_SEEDS:
        raise RuntimeError(
            f"Seed tidak sesuai: "
            f"{sorted(df['seed'].astype(int).unique())}"
        )

    if df[
        ["row_id", "candidate_index", "seed"]
    ].duplicated().any():
        raise RuntimeError(
            "Duplikasi row_id/candidate_index/seed."
        )

    for row_id, group in df.groupby(
        "row_id",
        sort=False,
    ):
        if set(
            group["candidate_index"].astype(int)
        ) != {1, 2, 3, 4, 5}:
            raise RuntimeError(
                f"{row_id}: candidate_index bukan 1..5."
            )

    if "temperature" in df.columns:
        vals = pd.to_numeric(
            df["temperature"],
            errors="coerce",
        ).dropna()
        if len(vals) and not np.allclose(
            vals.astype(float),
            0.70,
            atol=1e-12,
            rtol=0,
        ):
            raise RuntimeError(
                "temperature bukan 0.70."
            )

    if "top_p" in df.columns:
        vals = pd.to_numeric(
            df["top_p"],
            errors="coerce",
        ).dropna()
        if len(vals) and not np.allclose(
            vals.astype(float),
            0.95,
            atol=1e-12,
            rtol=0,
        ):
            raise RuntimeError(
                "top_p bukan 0.95."
            )

    if "top_k" in df.columns:
        vals = pd.to_numeric(
            df["top_k"],
            errors="coerce",
        ).dropna()
        if len(vals) and not np.allclose(
            vals.astype(float),
            0.0,
            atol=1e-12,
            rtol=0,
        ):
            raise RuntimeError(
                "top_k bukan 0."
            )

    if "max_new_tokens" in df.columns:
        vals = pd.to_numeric(
            df["max_new_tokens"],
            errors="coerce",
        ).dropna()
        if len(vals) and not np.allclose(
            vals.astype(float),
            128.0,
            atol=1e-12,
            rtol=0,
        ):
            raise RuntimeError(
                "max_new_tokens bukan 128."
            )


candidate_paths = sorted(
    INPUT_SCENARIO_DIR.glob(
        "run_*/candidates_long.csv"
    ),
    key=lambda p: p.stat().st_mtime,
    reverse=True,
)

valid_inputs = []

for path in candidate_paths:
    try:
        df_try = canonicalize(path)
        audit_candidates(df_try)

        scenario_values = set(
            df_try["scenario"].astype(str).unique()
        )
        if scenario_values != {"fsp_locked_exact"}:
            raise RuntimeError(
                f"Scenario tidak sesuai: {scenario_values}"
            )

        run_dir = path.parent
        manifest = (
            read_json_if_exists(
                run_dir / "run_manifest.json"
            )
            or read_json_if_exists(
                run_dir / "manifest.json"
            )
        )

        if manifest is None:
            raise RuntimeError(
                "Manifest generation tidak ditemukan."
            )

        ref_flag = manifest.get(
            "reference_read",
            manifest.get(
                "test_reference_read",
                None,
            ),
        )
        if ref_flag is not False:
            raise RuntimeError(
                "Manifest tidak membuktikan reference_read=False."
            )

        seeds_manifest = manifest.get(
            "candidate_seeds",
            manifest.get("seeds", []),
        )
        if set(map(int, seeds_manifest)) != EXPECTED_SEEDS:
            raise RuntimeError(
                f"Seed manifest tidak sesuai: {seeds_manifest}"
            )

        gen = manifest.get("generation", {})
        if not np.isclose(
            float(gen.get("temperature", -1)),
            0.70,
        ):
            raise RuntimeError(
                "Manifest temperature bukan 0.70."
            )
        if not np.isclose(
            float(gen.get("top_p", -1)),
            0.95,
        ):
            raise RuntimeError(
                "Manifest top_p bukan 0.95."
            )
        if int(gen.get("top_k", -999)) != 0:
            raise RuntimeError(
                "Manifest top_k bukan 0."
            )
        if int(
            gen.get("max_new_tokens", -1)
        ) != 128:
            raise RuntimeError(
                "Manifest max_new_tokens bukan 128."
            )
        if (
            "num_beams" in gen
            and int(gen["num_beams"]) != 1
        ):
            raise RuntimeError(
                "Manifest num_beams bukan 1."
            )

        valid_inputs.append(
            (path, df_try, manifest)
        )

    except Exception as exc:
        print(
            "Skip:",
            path,
            "|",
            type(exc).__name__,
            str(exc)[:220],
        )

if not valid_inputs:
    raise FileNotFoundError(
        "Tidak ditemukan final candidates_long.csv "
        "yang lolos audit."
    )

CANDIDATES_PATH, candidates, generation_manifest = (
    valid_inputs[0]
)
INPUT_RUN_DIR = CANDIDATES_PATH.parent
INPUT_RUN_NAME = INPUT_RUN_DIR.name
OUTPUT_DIR = OUTPUT_ROOT / INPUT_RUN_NAME
OUTPUT_DIR.mkdir(
    parents=True,
    exist_ok=True,
)

print("\nINPUT FINAL TERPILIH")
print("Candidates :", CANDIDATES_PATH)
print("Run        :", INPUT_RUN_NAME)
print("Rows       :", len(candidates))
print("Unique test:", candidates["row_id"].nunique())
print("Reference  : TIDAK ADA / TIDAK DIBACA")
print("Output dir :", OUTPUT_DIR)

display(candidates.head(10))


INPUT FINAL TERPILIH
Candidates : /workspace/10122233_CikitaMeydilvira_Skripsi/Hasil/final_generation/fsp_locked/run_14c344bdb72c6065/candidates_long.csv
Run        : run_14c344bdb72c6065
Rows       : 265
Unique test: 53
Reference  : TIDAK ADA / TIDAK DIBACA
Output dir : /workspace/10122233_CikitaMeydilvira_Skripsi/Hasil/final_selection/fsp_locked/run_14c344bdb72c6065


,scenario,row_id,conversation_no,turn_order,source_text,candidate_index,seed,candidate_text,technical_error,temperature,...,top_k,max_new_tokens,coverage,fragment_count,example_count,max_fragment_len,k_per_fragment,prompt_tokens,completion_tokens,latency_seconds
0,fsp_locked_exact,C03_T01_A1,3,1,Denmoraka naden tamale tama Panyurak?,1,42,Pernahkah ada seekor kambing yang berlari di k...,,0.7,...,0,128,0.6,3,7,7,6,530,27,5.275099
1,fsp_locked_exact,C03_T01_A1,3,1,Denmoraka naden tamale tama Panyurak?,2,43,Pernahkah ada kerbau yang memakan pohon panyurak?,,0.7,...,0,128,0.6,3,7,7,6,530,18,1.110049
2,fsp_locked_exact,C03_T01_A1,3,1,Denmoraka naden tamale tama Panyurak?,3,44,Pernahkah ada kucing yang tidur di atas atap r...,,0.7,...,0,128,0.6,3,7,7,6,530,18,1.061120
3,fsp_locked_exact,C03_T01_A1,3,1,Denmoraka naden tamale tama Panyurak?,4,45,Pernahkah ada anak yang bermain di sungai?,,0.7,...,0,128,0.6,3,7,7,6,530,14,0.883955
4,fsp_locked_exact,C03_T01_A1,3,1,Denmoraka naden tamale tama Panyurak?,5,46,Pernahkah ada orang yang datang ke kebun saya?,,0.7,...,0,128,0.6,3,7,7,6,530,16,0.936797
5,fsp_locked_exact,C03_T02_B1,3,2,"Ieq, penduanmoq.",1,42,"Ya, terjemahannya adalah ""Ya, saya baik-baik s...",,0.7,...,0,128,0.5,1,6,7,6,549,18,1.088867
6,fsp_locked_exact,C03_T02_B1,3,2,"Ieq, penduanmoq.",2,43,"Ya, sudah dipelajari.",,0.7,...,0,128,0.5,1,6,7,6,549,9,0.533834
7,fsp_locked_exact,C03_T02_B1,3,2,"Ieq, penduanmoq.",3,44,"Ya, aku sudah tahu.",,0.7,...,0,128,0.5,1,6,7,6,549,8,0.463044
8,fsp_locked_exact,C03_T02_B1,3,2,"Ieq, penduanmoq.",4,45,"Ya, sudah tahu.",,0.7,...,0,128,0.5,1,6,7,6,549,7,0.421096
9,fsp_locked_exact,C03_T02_B1,3,2,"Ieq, penduanmoq.",5,46,"Ya, sudah dipelajari.",,0.7,...,0,128,0.5,1,6,7,6,549,9,0.546511


## 4. Validasi kandidat

In [4]:
REFUSAL_PATTERNS = [
    r"^maaf\b",
    r"^mohon maaf\b",
    r"\bsaya tidak (mengerti|memahami|dapat|bisa)\b",
    r"\btidak dapat menerjemahkan\b",
    r"\btidak bisa menerjemahkan\b",
    r"\bi cannot\b",
    r"\bi can't\b",
]

def candidate_validity(
    text: str,
    technical_error: str,
):
    if str(technical_error).strip():
        return "invalid", "technical_failure"

    normalized = re.sub(
        r"\s+",
        " ",
        str(text).strip(),
    )

    if len(normalized) < 2:
        return "invalid", "empty_or_too_short"

    low = normalized.casefold()

    for pattern in REFUSAL_PATTERNS:
        if re.search(
            pattern,
            low,
            flags=re.IGNORECASE,
        ):
            return "invalid", "refusal"

    return "valid", ""


results = candidates.apply(
    lambda r: candidate_validity(
        r["candidate_text"],
        r["technical_error"],
    ),
    axis=1,
)

candidates["semantic_status"] = [
    x[0] for x in results
]
candidates["semantic_reason"] = [
    x[1] for x in results
]

print(
    candidates[
        ["semantic_status", "semantic_reason"]
    ].value_counts(dropna=False)
)

invalid_preview = candidates[
    candidates["semantic_status"] != "valid"
].copy()

if not invalid_preview.empty:
    display(
        invalid_preview[
            [
                "row_id",
                "candidate_index",
                "seed",
                "candidate_text",
                "semantic_reason",
            ]
        ].head(30)
    )
else:
    print("Semua 265 kandidat valid.")

semantic_status  semantic_reason
valid                               262
invalid          refusal              3
Name: count, dtype: int64


,row_id,candidate_index,seed,candidate_text,semantic_reason
51,C03_T11_A6,2,43,Saya tidak dapat menemukan contoh teks bahasa ...,refusal
181,C09_T22_B11,2,43,"Saya tidak mengerti apa itu ""Duri"" bahasa.",refusal
260,C33_T16_B8,1,42,"Karena terjemahan tidak diberikan, saya tidak ...",refusal


## 5. Load MPNet

In [5]:
print(
    "Load embedding model:",
    EMBEDDING_MODEL_ID,
)

embedder = SentenceTransformer(
    EMBEDDING_MODEL_ID,
    device=EMBEDDING_DEVICE,
    cache_folder=str(EMBEDDING_CACHE),
    trust_remote_code=False,
)

probe = embedder.encode(
    ["Ini kalimat uji."],
    convert_to_numpy=True,
    normalize_embeddings=True,
    show_progress_bar=False,
)

if probe.shape != (1, 768):
    raise RuntimeError(
        f"Dimensi embedding tidak sesuai: {probe.shape}"
    )

probe_norm = float(
    np.linalg.norm(probe[0])
)

if not np.isclose(
    probe_norm,
    1.0,
    atol=1e-4,
):
    raise RuntimeError(
        f"Embedding belum L2-normalized: {probe_norm}"
    )

print("Embedding dim : 768")
print("Probe norm    :", probe_norm)
print("EMBEDDING MODEL: READY")

Load embedding model: sentence-transformers/paraphrase-multilingual-mpnet-base-v2
Embedding dim : 768
Probe norm    : 1.0
EMBEDDING MODEL: READY


## 6. Hitung similarity, S3E, dan MBR

In [6]:
detail_rows = []
selected_rows = []
pair_rows = []
matrix_rows = []

row_order = (
    candidates[["row_id"]]
    .drop_duplicates()["row_id"]
    .tolist()
)

for row_id in row_order:
    group = (
        candidates[
            candidates["row_id"] == row_id
        ]
        .copy()
        .sort_values(
            "candidate_index",
            kind="stable",
        )
    )

    valid = (
        group[
            group["semantic_status"] == "valid"
        ]
        .copy()
        .sort_values(
            "candidate_index",
            kind="stable",
        )
    )

    n = len(valid)
    utilities = {}
    s3e_value = np.nan
    selected_candidate_index = np.nan
    selected_output = ""
    selected_utility = np.nan

    if n >= 2:
        texts = valid[
            "candidate_text"
        ].astype(str).tolist()

        E = embedder.encode(
            texts,
            batch_size=EMBEDDING_BATCH_SIZE,
            convert_to_numpy=True,
            normalize_embeddings=True,
            show_progress_bar=False,
        ).astype(np.float64)

        norms = np.linalg.norm(E, axis=1)
        if not np.allclose(
            norms,
            1.0,
            atol=1e-4,
        ):
            raise RuntimeError(
                f"{row_id}: embedding tidak normalized."
            )

        sim = np.clip(
            E @ E.T,
            0.0,
            1.0,
        )
        np.fill_diagonal(sim, 1.0)

        sim_off = sim.copy()
        np.fill_diagonal(sim_off, 0.0)

        mean_off = (
            sim_off.sum(axis=1)
            / (n - 1)
        )

        s3e_value = float(
            -np.mean(
                np.log(
                    np.maximum(
                        mean_off,
                        EPS,
                    )
                )
            )
        )

        valid_indices = (
            valid["candidate_index"]
            .astype(int)
            .tolist()
        )

        utilities = {
            int(valid_indices[i]):
                float(mean_off[i])
            for i in range(n)
        }

        best_utility = max(
            utilities.values()
        )
        tied = [
            idx
            for idx, utility
            in utilities.items()
            if abs(
                utility - best_utility
            ) <= TIE_TOL
        ]
        selected_candidate_index = int(
            min(tied)
        )

        selected_row = valid[
            valid["candidate_index"]
            == selected_candidate_index
        ].iloc[0]

        selected_output = str(
            selected_row["candidate_text"]
        )
        selected_utility = float(
            utilities[
                selected_candidate_index
            ]
        )
        selection_status = "mbr_selected"

        for i, ri in enumerate(
            valid.itertuples(index=False)
        ):
            for j, rj in enumerate(
                valid.itertuples(index=False)
            ):
                pair_rows.append({
                    "scenario": SCENARIO_SLUG,
                    "row_id": str(row_id),
                    "candidate_i":
                        int(ri.candidate_index),
                    "candidate_j":
                        int(rj.candidate_index),
                    "is_diagonal":
                        bool(i == j),
                    "cosine_similarity":
                        float(sim[i, j]),
                })

        matrix_rows.append({
            "scenario": SCENARIO_SLUG,
            "row_id": str(row_id),
            "valid_candidate_indices":
                valid_indices,
            "similarity_matrix":
                sim.tolist(),
        })

    elif n == 1:
        only = valid.iloc[0]
        selected_candidate_index = int(
            only["candidate_index"]
        )
        selected_output = str(
            only["candidate_text"]
        )
        selection_status = (
            "single_valid_candidate"
        )
        matrix_rows.append({
            "scenario": SCENARIO_SLUG,
            "row_id": str(row_id),
            "valid_candidate_indices":
                [selected_candidate_index],
            "similarity_matrix": None,
        })

    else:
        selection_status = "inference_failure"
        matrix_rows.append({
            "scenario": SCENARIO_SLUG,
            "row_id": str(row_id),
            "valid_candidate_indices": [],
            "similarity_matrix": None,
        })

    for _, r in group.iterrows():
        idx = int(r["candidate_index"])

        detail = {
            "scenario": SCENARIO_SLUG,
            "row_id": str(row_id),
            "conversation_no":
                r.get("conversation_no", ""),
            "turn_order":
                r.get("turn_order", ""),
            "source_text":
                str(r["source_text"]),
            "candidate_index": idx,
            "candidate_id": f"C{idx}",
            "seed": int(r["seed"]),
            "candidate_text":
                str(r["candidate_text"]),
            "technical_error":
                str(r["technical_error"]),
            "semantic_status":
                str(r["semantic_status"]),
            "semantic_reason":
                str(r["semantic_reason"]),
            "mbr_utility":
                utilities.get(idx, np.nan),
            "is_selected_by_mbr":
                bool(
                    selection_status
                    == "mbr_selected"
                    and idx
                    == int(
                        selected_candidate_index
                    )
                ),
            "s3e_value": s3e_value,
            "valid_candidate_count": int(n),
            "selection_status":
                selection_status,
        }

        for col in [
            "coverage",
            "fragment_count",
            "example_count",
            "max_fragment_len",
            "k_per_fragment",
            "prompt_tokens",
            "completion_tokens",
            "latency_seconds",
            "generation_time_seconds",
        ]:
            if col in r.index:
                detail[col] = r[col]

        detail_rows.append(detail)

    base = group.iloc[0]

    final_row = {
        "scenario": SCENARIO_SLUG,
        "row_id": str(row_id),
        "conversation_no":
            base.get("conversation_no", ""),
        "turn_order":
            base.get("turn_order", ""),
        "source_text":
            str(base["source_text"]),
        "selected_candidate_index":
            selected_candidate_index,
        "selected_candidate":
            ""
            if pd.isna(
                selected_candidate_index
            )
            else f"C{int(selected_candidate_index)}",
        "selected_output": selected_output,
        "selected_utility": selected_utility,
        "s3e_value": s3e_value,
        "valid_candidate_count": int(n),
        "selection_status":
            selection_status,
    }

    for col in [
        "coverage",
        "fragment_count",
        "example_count",
        "max_fragment_len",
        "k_per_fragment",
    ]:
        if col in base.index:
            final_row[col] = base[col]

    selected_rows.append(final_row)


details_df = pd.DataFrame(detail_rows)
selected_df = pd.DataFrame(selected_rows)
pairs_df = pd.DataFrame(pair_rows)

if len(details_df) != 265:
    raise RuntimeError(
        f"Detail rows {len(details_df)} != 265."
    )

if len(selected_df) != 53:
    raise RuntimeError(
        f"Final rows {len(selected_df)} != 53."
    )

print("Candidate detail rows:", len(details_df))
print("Final group rows      :", len(selected_df))
print("\nSelection status:")
print(
    selected_df[
        "selection_status"
    ].value_counts(dropna=False)
)

display(selected_df.head(10))

Candidate detail rows: 265
Final group rows      : 53

Selection status:
selection_status
mbr_selected    53
Name: count, dtype: int64


,scenario,row_id,conversation_no,turn_order,source_text,selected_candidate_index,selected_candidate,selected_output,selected_utility,s3e_value,valid_candidate_count,selection_status,coverage,fragment_count,example_count,max_fragment_len,k_per_fragment
0,fsp_locked,C03_T01_A1,3,1,Denmoraka naden tamale tama Panyurak?,1,C1,Pernahkah ada seekor kambing yang berlari di k...,0.558616,7.408302e-01,5,mbr_selected,0.600000,3,7,7,6
1,fsp_locked,C03_T02_B1,3,2,"Ieq, penduanmoq.",2,C2,"Ya, sudah dipelajari.",0.827185,2.735338e-01,5,mbr_selected,0.500000,1,6,7,6
2,fsp_locked,C03_T03_A2,3,3,Piranra tacappaq male tama Panyurak?,5,C5,Kapan baru pergi ke Panyurak?,0.641883,5.086425e-01,5,mbr_selected,0.400000,2,6,7,6
3,fsp_locked,C03_T04_B2,3,4,Ia tonna bulan dua tanggalaq duangpulo annan.,1,C1,Bulan dua tanggal dua puluh.,1.000000,2.717683e-09,5,mbr_selected,1.000000,5,12,7,6
4,fsp_locked,C03_T05_A3,3,5,Mangngaparikiq male tama?,2,C2,Apa yang dikerjakan di rumah? Apa yang dikerja...,0.558573,8.355223e-01,5,mbr_selected,0.666667,2,8,7,6
5,fsp_locked,C03_T06_B3,3,6,Malenaq ngkita bola loqbang.,2,C2,Saya pergi ke pasar.,0.850877,3.107005e-01,5,mbr_selected,0.500000,2,9,7,6
6,fsp_locked,C03_T07_A4,3,7,Ciapari naloqbang joq bola?,3,C3,Mengapa rumah itu runtuh?,0.834481,2.365841e-01,5,mbr_selected,0.750000,3,15,7,6
7,fsp_locked,C03_T08_B4,3,8,"Taqdei puangna, sanga torro jiongri Pinrang.",3,C3,"Hari ini sore, karena kamu tinggal di Pinrang.",0.531342,9.829432e-01,5,mbr_selected,0.333333,2,9,7,6
8,fsp_locked,C03_T09_A5,3,9,Makassingraka joq bola?,5,C5,Siapakah orang-orang di rumah?,0.707455,5.661558e-01,5,mbr_selected,0.666667,2,10,7,6
9,fsp_locked,C03_T10_B5,3,10,"Makassing na matonggo, pa masussa to wai.",3,C3,"Sudah tiba waktunya, mana yang lebih sulit?",0.367758,1.328715e+00,5,mbr_selected,0.714286,4,10,7,6


## 7. Simpan artefak

In [7]:
DETAIL_PATH = (
    OUTPUT_DIR
    / "candidate_s3e_mbr_details.csv"
)
SELECTED_PATH = (
    OUTPUT_DIR
    / "mbr_selected.csv"
)
PAIR_PATH = (
    OUTPUT_DIR
    / "similarity_pairs.csv"
)
MATRIX_PATH = (
    OUTPUT_DIR
    / "similarity_matrices.jsonl"
)
INVALID_PATH = (
    OUTPUT_DIR
    / "invalid_candidates.csv"
)
SUMMARY_PATH = (
    OUTPUT_DIR
    / "s3e_mbr_summary.json"
)
MANIFEST_PATH = (
    OUTPUT_DIR
    / "processing_manifest.json"
)

details_df.to_csv(
    DETAIL_PATH,
    index=False,
    encoding="utf-8-sig",
)
selected_df.to_csv(
    SELECTED_PATH,
    index=False,
    encoding="utf-8-sig",
)
pairs_df.to_csv(
    PAIR_PATH,
    index=False,
    encoding="utf-8-sig",
)

invalid_df = details_df[
    details_df["semantic_status"] != "valid"
].copy()

invalid_df.to_csv(
    INVALID_PATH,
    index=False,
    encoding="utf-8-sig",
)

with MATRIX_PATH.open(
    "w",
    encoding="utf-8",
) as f:
    for record in matrix_rows:
        f.write(
            json.dumps(
                record,
                ensure_ascii=False,
            )
            + "\n"
        )

s3e_series = selected_df[
    "s3e_value"
].dropna()

summary = {
    "created_at_utc":
        datetime.now(
            timezone.utc
        ).isoformat(),
    "scenario": SCENARIO_SLUG,
    "scenario_label": SCENARIO_LABEL,
    "input_candidates_path":
        str(CANDIDATES_PATH),
    "input_run_name":
        INPUT_RUN_NAME,
    "candidate_rows":
        int(len(details_df)),
    "test_groups":
        int(len(selected_df)),
    "mbr_selected_groups":
        int(
            (
                selected_df[
                    "selection_status"
                ]
                == "mbr_selected"
            ).sum()
        ),
    "single_valid_candidate_groups":
        int(
            (
                selected_df[
                    "selection_status"
                ]
                == "single_valid_candidate"
            ).sum()
        ),
    "inference_failure_groups":
        int(
            (
                selected_df[
                    "selection_status"
                ]
                == "inference_failure"
            ).sum()
        ),
    "valid_candidates":
        int(
            (
                details_df[
                    "semantic_status"
                ]
                == "valid"
            ).sum()
        ),
    "invalid_candidates":
        int(len(invalid_df)),
    "s3e_groups":
        int(len(s3e_series)),
    "s3e_mean":
        float(s3e_series.mean())
        if len(s3e_series)
        else None,
    "s3e_median":
        float(s3e_series.median())
        if len(s3e_series)
        else None,
    "s3e_std":
        float(s3e_series.std(ddof=1))
        if len(s3e_series) > 1
        else None,
    "s3e_min":
        float(s3e_series.min())
        if len(s3e_series)
        else None,
    "s3e_max":
        float(s3e_series.max())
        if len(s3e_series)
        else None,
    "embedding_model":
        EMBEDDING_MODEL_ID,
    "embedding_dimension": 768,
    "embedding_normalized": True,
    "similarity":
        "cosine clipped to [0,1]",
    "s3e_estimator":
        "leave-one-out mean off-diagonal similarity",
    "s3e_epsilon": EPS,
    "mbr_utility":
        "mean off-diagonal MPNet cosine",
    "mbr_tie_break":
        "smallest candidate_index within tolerance 1e-12",
    "reference_used": False,
}

SUMMARY_PATH.write_text(
    json.dumps(
        summary,
        ensure_ascii=False,
        indent=2,
    ),
    encoding="utf-8",
)

manifest = {
    "created_at_utc":
        datetime.now(
            timezone.utc
        ).isoformat(),
    "scenario": SCENARIO_SLUG,
    "input_path":
        str(CANDIDATES_PATH),
    "input_sha256":
        hashlib.sha256(
            CANDIDATES_PATH.read_bytes()
        ).hexdigest(),
    "generation_manifest":
        generation_manifest,
    "reference_columns_forbidden":
        sorted(
            FORBIDDEN_REFERENCE_COLUMNS
        ),
    "reference_used": False,
    "embedding_model":
        EMBEDDING_MODEL_ID,
    "embedding_device":
        EMBEDDING_DEVICE,
    "embedding_cache":
        str(EMBEDDING_CACHE),
    "python":
        platform.python_version(),
    "torch":
        torch.__version__,
    "sentence_transformers":
        __import__(
            "sentence_transformers"
        ).__version__,
    "numpy": np.__version__,
    "pandas": pd.__version__,
    "outputs": {
        "candidate_details":
            str(DETAIL_PATH),
        "mbr_selected":
            str(SELECTED_PATH),
        "similarity_pairs":
            str(PAIR_PATH),
        "similarity_matrices":
            str(MATRIX_PATH),
        "invalid_candidates":
            str(INVALID_PATH),
        "summary":
            str(SUMMARY_PATH),
    },
}

MANIFEST_PATH.write_text(
    json.dumps(
        manifest,
        ensure_ascii=False,
        indent=2,
    ),
    encoding="utf-8",
)

print(
    json.dumps(
        summary,
        ensure_ascii=False,
        indent=2,
    )
)

{
  "created_at_utc": "2026-08-13T19:01:37.638201+00:00",
  "scenario": "fsp_locked",
  "scenario_label": "Fragment-Shot Prompting (Exact Locked)",
  "input_candidates_path": "/workspace/10122233_CikitaMeydilvira_Skripsi/Hasil/final_generation/fsp_locked/run_14c344bdb72c6065/candidates_long.csv",
  "input_run_name": "run_14c344bdb72c6065",
  "candidate_rows": 265,
  "test_groups": 53,
  "mbr_selected_groups": 53,
  "single_valid_candidate_groups": 0,
  "inference_failure_groups": 0,
  "valid_candidates": 262,
  "invalid_candidates": 3,
  "s3e_groups": 53,
  "s3e_mean": 0.6912426212440577,
  "s3e_median": 0.7184265325035938,
  "s3e_std": 0.3844559905254037,
  "s3e_min": 2.7176827492051064e-09,
  "s3e_max": 1.7549394329176309,
  "embedding_model": "sentence-transformers/paraphrase-multilingual-mpnet-base-v2",
  "embedding_dimension": 768,
  "embedding_normalized": true,
  "similarity": "cosine clipped to [0,1]",
  "s3e_estimator": "leave-one-out mean off-diagonal similarity",
  "s3e_epsi

## 8. Final audit + cleanup

In [8]:
checks = {
    "candidate_rows_265":
        len(details_df) == 265,
    "groups_53":
        len(selected_df) == 53,
    "five_candidates_each":
        bool(
            (
                details_df
                .groupby("row_id")
                .size()
                == 5
            ).all()
        ),
    "reference_not_in_input":
        not bool(
            FORBIDDEN_REFERENCE_COLUMNS
            & set(
                pd.read_csv(
                    CANDIDATES_PATH,
                    nrows=0,
                ).columns
            )
        ),
    "selected_exists":
        SELECTED_PATH.exists(),
    "details_exists":
        DETAIL_PATH.exists(),
    "matrices_exists":
        MATRIX_PATH.exists(),
    "summary_exists":
        SUMMARY_PATH.exists(),
    "manifest_exists":
        MANIFEST_PATH.exists(),
}

failed = [
    name
    for name, ok in checks.items()
    if not bool(ok)
]

display(
    pd.DataFrame(
        [
            {
                "check": name,
                "ok": bool(ok),
            }
            for name, ok in checks.items()
        ]
    )
)

if failed:
    raise RuntimeError(
        f"FINAL AUDIT GAGAL: {failed}"
    )

try:
    del embedder
except Exception:
    pass

gc.collect()

if torch.cuda.is_available():
    torch.cuda.empty_cache()
    try:
        torch.cuda.ipc_collect()
    except Exception:
        pass

print("\n" + "=" * 76)
print(
    f"{SCENARIO_LABEL} — S3E + MBR: COMPLETE"
)
print("=" * 76)
print("Input       :", CANDIDATES_PATH)
print("Selected    :", SELECTED_PATH)
print("Details     :", DETAIL_PATH)
print("S3E summary :", SUMMARY_PATH)
print("Reference   : TIDAK DIBACA")
print(
    "\nSetelah 05A, 05B, dan 05C selesai, "
    "baru lanjut evaluasi BLEU/chrF++ dengan Dataset/test.csv."
)

,check,ok
0,candidate_rows_265,True
1,groups_53,True
2,five_candidates_each,True
3,reference_not_in_input,True
4,selected_exists,True
5,details_exists,True
6,matrices_exists,True
7,summary_exists,True
8,manifest_exists,True



Fragment-Shot Prompting (Exact Locked) — S3E + MBR: COMPLETE
Input       : /workspace/10122233_CikitaMeydilvira_Skripsi/Hasil/final_generation/fsp_locked/run_14c344bdb72c6065/candidates_long.csv
Selected    : /workspace/10122233_CikitaMeydilvira_Skripsi/Hasil/final_selection/fsp_locked/run_14c344bdb72c6065/mbr_selected.csv
Details     : /workspace/10122233_CikitaMeydilvira_Skripsi/Hasil/final_selection/fsp_locked/run_14c344bdb72c6065/candidate_s3e_mbr_details.csv
S3E summary : /workspace/10122233_CikitaMeydilvira_Skripsi/Hasil/final_selection/fsp_locked/run_14c344bdb72c6065/s3e_mbr_summary.json
Reference   : TIDAK DIBACA

Setelah 05A, 05B, dan 05C selesai, baru lanjut evaluasi BLEU/chrF++ dengan Dataset/test.csv.
